[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/registration/registration.ipynb)

# Point-Cloud Registration

Point-cloud registration finds the rigid motion that carries one set of points onto another, given which point goes where: a scan of an object taken from two places, or a part located against its model. The two sets differ by measurement noise, so the best motion is the one that brings them closest.

A rigid motion is a motor, and a motor moves a point by its sandwich, `motor >> point`. Two fits run side by side. The first centres both sets, fits the rotation by making the turned displacements agree as well as they can, a form on rotors whose largest eigenvector is the rotation, and then moves the turned centre onto the target's. The second writes the motion as a linear equation, `target * Motor - Motor * source`, a map on motors that vanishes on the true motion, and fits rotation and translation at once as the smallest eigenvector of its squared size. The first minimizes the summed squared distances between the points; the second the squared size of the equation's coefficients, so it can never do better on distances, and here does as well to many digits.

In the matrix notation of registration the first fit reads as the Kabsch algorithm, a singular value decomposition of the cross-covariance $\sum_k \mathbf y_k \mathbf x_k^{\mathsf T}$ of the centred points, and the second as the eigenvector method of Horn, with quaternions in place of the rotor.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import stack
from numga import NumpyContext
from numga.algebras import PGA3D as ga
from examples.animation import save_figure
from examples.geometry.registration import render
from examples.geometry.registration.core import cloud, jitter

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()
Motor = ga.gatype.rotor()
# The Euclidean rotation subgroup, fixing the chosen PGA origin.
Rotor = ga.gatype.even().derive.nondegenerate
Vector = ga.gatype.vector().derive.nondegenerate

## 1. Noisy correspondences

The source is a cloud of points spread unevenly along x, y and z. The target is the source moved by a known motion, a shift after a turn, with each point then nudged by a little noise.

In [ ]:
# The source's points, their spreads along x, y and z and the seed of the noise; the true motion's shift and
# turn; and the noise that moves each target point.
n_points, cloud_spread, seed = 60, np.array([2.0, 1.0, 0.5]), 0
shift, turn = np.array([1.5, -0.5, 2.0]), np.array([0.8, -0.6, 1.4])
noise = 0.02

In [ ]:
rng = np.random.default_rng(seed)
source = cloud(n_points, cloud_spread, rng)                                    # [points] Point
true_motor = ((stack((mv.xw, mv.yw, mv.zw)) * shift).sum(axis=0) / 2).exp() * ((stack((mv.xy, mv.yz, mv.zx)) * turn).sum(axis=0) / 2).exp()
target = jitter(true_motor >> source, noise, rng)                              # [points] Point

## 2. Centring, then turning

Both sets are centred on their means; the differences of points are directions, read as Euclidean vectors through their duals. The rotation that best turns the source's displacements onto the target's maximizes their summed scalar products, a form on rotors, at its largest eigenvector. The translation then carries the turned source centre onto the target's: a product of two point reflections translates by twice their separation, so its square root moves one point onto the other. The source, grey, the target, blue crosses, and the aligned source, red, each tied to its target.

In [ ]:
def fit_rotor(source: Vector, target: Vector) -> Rotor:
    """Maximize sandwich alignment of corresponding Euclidean vectors."""
    alignment = target.scalar_product(Rotor >> source).sum(axis=0)
    values, rotors = alignment.eigh()
    return rotors[values.argmax()].normalized()


def fit_motor_alignment(source: Point, target: Point) -> Motor:
    """Fit a rigid pose by centered alignment and centroid matching."""
    source, target = source.normalized(), target.normalized()
    source_mean, target_mean = source.mean(axis=0), target.mean(axis=0)

    # Point differences are ideal points. Their duals carry the Euclidean
    # displacements; the spatial part discards the homogeneous weight component.
    source_vectors = (source - source_mean).dual().select_subspace(Vector.output_subspace)
    target_vectors = (target - target_mean).dual().select_subspace(Vector.output_subspace)
    rotation = fit_rotor(source_vectors, target_vectors)

    # A product of point reflections translates by twice their separation.
    # Its square root carries the rotated source centroid onto the target's.
    translation = (target_mean * (rotation >> source_mean).inverse()).square_root()
    return translation * rotation


sandwich = fit_motor_alignment(source, target)                                 # [] Motor
save_figure(render.draw_registration(source, target, sandwich >> source), "registration_sandwich_alignment");

## 3. One equation for the whole motion

A motor that carries each source point onto its target satisfies `target * motor == motor * source`, linear in the motor. The squared size of that residual, summed over the points, is a form on motors; its bulk sees rotation and its weight, the bulk of the complement, sees translation, so both are fitted at once. Measured against the motor's own metric, which sees only the rotation, the flattest direction of the form is the fitted motor.

In [ ]:
def fit_motor(source: Point, target: Point) -> Motor:
    """Fit a motor by the coefficient residual of the one-sided equations."""
    source, target = source.normalized(), target.normalized()
    # target * Motor == Motor * source is linear in the motor: the residual is a map on motors.
    residual = target * Motor - Motor * source

    # The bulk norm alone would discard translation: the PGA scalar product is degenerate on
    # ideal blades. Adding the weight norm, the bulk norm of the complement, sums the squares
    # of every coefficient, retaining errors in ideal components too.
    bulk = residual.scalar_norm_squared()
    weight = residual.dual().scalar_norm_squared()
    misfit = (bulk + weight).sum(axis=0)
    # Against the motor's own metric, the scalar part of motor * motor.reverse(), only the rotor
    # coefficients are measured: translation carries no unit of its own, so the fit does not depend
    # on where the origin sits or on the scene's scale. The metric is blind to translation, so for
    # each rotor mode eigh chooses the translation that minimizes the misfit, and normalized()
    # then makes motor * motor.reverse() == 1, its pseudoscalar part included.
    values, motors = misfit.eigh()
    return motors[values.argmin()].normalized()


one_sided = fit_motor(source, target)                                          # [] Motor
save_figure(render.draw_registration(source, target, one_sided >> source), "registration_one_sided_residual");

In [ ]:
def squared_distances(points: Point, targets: Point):
    """The summed squared distances of points of unit weight from their targets: the squared norms of their
    differences' complements."""
    return ((points.normalized() - targets.normalized()).dual().norm_squared()).sum(axis=0)


cartesian = squared_distances(sandwich >> source, target)                     # [] Scalar
coefficient = squared_distances(one_sided >> source, target)                  # [] Scalar

In [ ]:
# checks
# Centring then aligning minimizes the summed squared distances, so the one-sided fit cannot beat it on that
# measure; both come within the noise of the target.
assert cartesian.to_array() <= coefficient.to_array() * (1 + 1e-12)
assert np.sqrt(coefficient.to_array() / n_points) < 3 * noise * np.sqrt(3)